# NBA Draft Position & Second Contract Salary
### DSO 579 — Advanced Sports Performance Analytics — Group Project

---

**Course-aligned methods (ISLR / DSO 579):**
Multiple Linear Regression (Ch. 3) · Decision Tree & Random Forest (Ch. 8) · k-Fold Cross Validation (Ch. 5) · K-Means Clustering for player profiles (Ch. 12)

**Notebook Outline:**
1. Introduction & Background  
2. Research Questions & Hypotheses *(primary + 4 sub-RQs)*  
3. Methods & Data  
4. Data Loading & Cleaning  
5. Descriptive Statistics  
6. Statistical Tests *(Pearson, Spearman, ANOVA)*  
7. Multiple Linear Regression *(statsmodels inference)*  
8. Decision Tree & Random Forest *(with k-fold CV)*  
9. Sub-RQ Analyses *(position, era, performance moderation)*  
10. K-Means Clustering — Player Archetypes  
11. Key Findings  
12. Implications & Recommendations

## 1. Introduction & Background

The NBA Draft is a primary mechanism for talent allocation in professional basketball. Under the current Collective Bargaining Agreement, all first-round picks sign a **rookie-scale contract** lasting four years (two guaranteed + two team-option years). Once that period ends, players negotiate their **second contract** — frequently the most consequential financial decision of their career.

Front offices invest heavily in scouting because high picks are widely believed to deliver outsized long-term value. But is that belief financially justified? Does the early pick *signal* a future financial commitment, or does on-court performance over the rookie deal override draft pedigree when negotiating the second contract?

**Why this matters:**
- **Front offices** plan multi-year cap strategy around expected re-signing costs.
- **Coaches** allocate developmental minutes, often (implicitly or explicitly) guided by draft pedigree.
- **Players & agents** benchmark expected market value against historical draft cohorts.

## 2. Research Questions & Hypotheses

### Primary Research Question
> **RQ:** Does a player's overall draft pick number predict the size of their second NBA contract, controlling for rookie-era performance?

**Hypotheses:**
- **H₁:** Lower pick numbers (earlier picks) are associated with higher second-contract salaries even after controlling for performance.
- **H₀:** Once performance is controlled, draft pick has no meaningful effect on second-contract salary.

### Sub Research Questions
| # | Question | Why it matters |
|---|---------|----------------|
| **SQ1** | Does the pick → salary relationship hold across **player positions** (Guard / Forward / Center)? | Markets value scarcity differently across positions. |
| **SQ2** | Does **rookie-era performance** *moderate* the effect of draft pick on second-contract pay? | Tells us whether high picks are paid for *production* or for *pedigree*. |
| **SQ3** | Has the relationship changed across **draft eras** (2010–2014 vs 2015–2021)? | The 2017 CBA changed cap dynamics — does the pick effect persist post-cap-spike? |
| **SQ4** | Can we identify **player archetypes** (e.g., "value picks" vs "draft busts") via clustering? | Operationalizes the GM's mental model of "hits and misses." |

### Supporting Literature
- *Massey & Thaler (2013)*: "The Loser's Curse" — early picks systematically overvalued in NFL drafts.  
- *Berri, Brook & Schmidt (2007)*: NBA decision-makers overweight draft position when valuing players.  
- *Coates & Oguntimein (2010)*: rookie-scale compresses pay relative to performance, predicting market correction at second contract.

## 3. Methods & Data

### Analytical Approach (course-aligned)
1. **Merge** draft data with year-by-year salary data via player name.
2. For each first-round pick (2010–2021), identify their salary in `draft_year + 4` — the **second-contract season**.
3. Normalize salary as **% of league cap** to control for cap inflation.
4. Test relationships via correlation, ANOVA, and regression.
5. Build models in increasing complexity (course coverage):
   - **Multiple Linear Regression** (ISLR Ch. 3) — interpretable baseline with p-values & confidence intervals.
   - **Decision Tree** (ISLR Ch. 8) — captures non-linear thresholds.
   - **Random Forest** (ISLR Ch. 8 ensembles) — variance reduction via bagging.
6. Use **5-fold Cross Validation** (ISLR Ch. 5) for honest model assessment.
7. Apply **K-Means Clustering** (ISLR Ch. 12) on standardized features to discover player archetypes.

### Data Sources
| Source | Coverage | Key Columns |
|--------|----------|-------------|
| Kaggle: `nbaplayersdraft.csv` (1989–2021) | Draft data | `year`, `overall_pick`, `player`, career stats |
| Kaggle: `NBA Player Stats and Salaries 2010–2025` | Annual salary + per-game stats | `Player`, `Year`, `Salary`, `Pos`, `PTS`, `AST`, `TRB` |

### Variables
- **IV:** `overall_pick` (1–30)
- **DV:** `salary_cap_pct` — second-contract salary as % of cap
- **Controls:** rookie-era career stats (PPG, RPG, APG, Win Shares, BPM, VORP), age in second-contract season

## 4. Data Loading & Cleaning

In [ ]:
draft = pd.read_csv('data/nbaplayersdraft.csv')
salary = pd.read_csv('data/NBA Player Stats and Salaries_2010-2025.csv')

print(f'Draft dataset:  {draft.shape[0]} rows × {draft.shape[1]} cols  ({draft.year.min()}–{draft.year.max()})')
print(f'Salary dataset: {salary.shape[0]} rows × {salary.shape[1]} cols  ({salary.Year.min()}–{salary.Year.max()})')

### Filter and identify second-contract salary

We restrict the sample to **first-round picks (overall_pick ≤ 30) drafted 2010–2021**:
- All players governed by the same rookie-scale contract structure.
- Their second-contract season (`draft_year + 4`) falls inside our salary window (2014–2025).

In [ ]:
# Filter to first-round picks 2010–2021
draft_1st = draft[
    (draft['overall_pick'] <= 30) &
    (draft['year'].between(2010, 2021))
].copy()
draft_1st['second_contract_year'] = draft_1st['year'] + 4
draft_1st = draft_1st.rename(columns={'player': 'Player', 'year': 'draft_year'})
print(f'First-round picks 2010–2021: {len(draft_1st)} players')

# Drop intra-season duplicates from trades (keep highest salary per player-year)
salary_clean = (salary
                .sort_values('Salary', ascending=False)
                .drop_duplicates(subset=['Player', 'Year']))

# Match each player to their second-contract-year salary
rows = []
for _, r in draft_1st.iterrows():
    m = salary_clean[(salary_clean['Player'] == r['Player']) &
                      (salary_clean['Year']   == r['second_contract_year'])]
    if not m.empty:
        s = m.iloc[0]
        rows.append({
            'Player': r['Player'], 'draft_year': r['draft_year'],
            'overall_pick': r['overall_pick'], 'second_contract_year': r['second_contract_year'],
            'second_contract_salary': s['Salary'], 'Pos': s['Pos'],
            'career_ppg': r['points_per_game'], 'career_rpg': r['average_total_rebounds'],
            'career_apg': r['average_assists'], 'career_ws': r['win_shares'],
            'career_bpm': r['box_plus_minus'], 'career_vorp': r['value_over_replacement'],
            'sc_age': s['Age'], 'sc_ppg': s['PTS'], 'sc_rpg': s['TRB'],
            'sc_apg': s['AST'], 'sc_games': s['G']
        })

df = pd.DataFrame(rows)
df = df[df['second_contract_salary'] > 100_000].copy()  # drop data errors

# Normalize salary by salary cap (handles inflation across years)
cap_by_year = {2014: 58.679e6, 2015: 63.065e6, 2016: 70.0e6, 2017: 99.093e6,
               2018: 99.093e6, 2019: 101.869e6, 2020: 109.140e6, 2021: 109.140e6,
               2022: 112.414e6, 2023: 123.655e6, 2024: 136.021e6, 2025: 140.588e6}
df['salary_cap_pct'] = df.apply(
    lambda r: r['second_contract_salary'] / cap_by_year.get(r['second_contract_year'], 120e6) * 100, axis=1)

# Position groupings (some rows have hybrid positions like "PG-SG")
def pos_group(p):
    if pd.isna(p): return 'Unknown'
    p = p.split('-')[0]
    if p in ['PG','SG']: return 'Guard'
    if p in ['SF','PF']: return 'Forward'
    if p == 'C': return 'Center'
    return 'Other'
df['pos_group'] = df['Pos'].apply(pos_group)

# Era groupings (CBA changed in 2017 with cap spike)
df['draft_era'] = np.where(df['draft_year'] <= 2014, '2010–2014', '2015–2021')

# Pick tiers for ANOVA
df['pick_tier'] = pd.cut(df['overall_pick'], bins=[0,5,10,15,20,30],
                          labels=['Top 5','6–10','11–15','16–20','21–30'])

print(f'Successfully matched: {len(df)} players')
print(f'Unmatched (no 2nd contract / out-of-league): {len(draft_1st) - len(df)}  ({(len(draft_1st)-len(df))/len(draft_1st)*100:.1f}%)')
df.head()

**Note on attrition:** ~27% of first-round picks have no salary in `draft_year + 4`. They were waived, played overseas, or did not survive the rookie-scale period. Including them as zero would conflate "no contract" with "low contract" — we treat this as **right-censored** data and analyze only players who reached a second contract.

## 5. Descriptive Statistics

In [ ]:
df[['overall_pick','second_contract_salary','salary_cap_pct']].describe().round(2)

In [ ]:
tier_stats = (df.groupby('pick_tier', observed=True)['salary_cap_pct']
                .agg(['mean','median','std','count']).round(2))
tier_stats

**Interpretation:** Top-5 picks earn ~8.0% of cap on their second deal — roughly **2.7×** what late first-rounders (21–30) earn (~2.9%). The pattern is monotonic: each tier earns less than the tier above.

In [ ]:
# Visualization 1: scatter + regression line
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(df['overall_pick'], df['salary_cap_pct'],
           alpha=0.5, s=60, edgecolors='steelblue', facecolors='lightblue')
m, b = np.polyfit(df['overall_pick'], df['salary_cap_pct'], 1)
r_corr, p_corr = stats.pearsonr(df['overall_pick'], df['salary_cap_pct'])
xs = np.linspace(1, 30, 100)
ax.plot(xs, m*xs + b, 'firebrick', lw=2, label=f'OLS fit: r = {r_corr:+.2f}, p < 0.001')
ax.set_xlabel('Overall Draft Pick'); ax.set_ylabel('2nd Contract Salary (% of Cap)')
ax.set_title('Draft Pick vs Second Contract Salary')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 筛选顺位在 20 到 25 之间的球员
tier_20_25 = df[(df['overall_pick'] >= 20) & (df['overall_pick'] <= 25)]

# 找出其中薪资占比最高的球员
the_outlier = tier_20_25.sort_values('salary_cap_pct', ascending=False).head(1)

print(the_outlier[['Player', 'overall_pick', 'salary_cap_pct', 'second_contract_year']])

In [ ]:
# Visualization 2: salary distribution by pick tier
order = ['Top 5','6–10','11–15','16–20','21–30']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='pick_tier', y='salary_cap_pct',
            order=order, ax=axes[0], palette='Blues_r')
axes[0].set(title='Salary Distribution by Pick Tier',
            xlabel='Pick Tier', ylabel='2nd Contract (% of Cap)')

means = df.groupby('pick_tier', observed=True)['salary_cap_pct'].mean().reindex(order)
bars = axes[1].bar(order, means.values, color=sns.color_palette('Blues_r', 5))
for bar, v in zip(bars, means.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.2,
                 f'{v:.1f}%', ha='center', fontsize=10)
axes[1].set(title='Mean 2nd Contract by Pick Tier',
            xlabel='Pick Tier', ylabel='Mean (% of Cap)')
plt.tight_layout(); plt.show()

## 6. Statistical Tests

In [ ]:
# Pearson — linear correlation
r, p = stats.pearsonr(df['overall_pick'], df['salary_cap_pct'])
print(f'Pearson r:   {r:+.3f}  (p = {p:.4g})')

# Spearman — rank-based, robust
rho, p2 = stats.spearmanr(df['overall_pick'], df['salary_cap_pct'])
print(f'Spearman ρ:  {rho:+.3f}  (p = {p2:.4g})')

# One-way ANOVA across pick tiers
groups = [g['salary_cap_pct'].values for _, g in df.groupby('pick_tier', observed=True)]
f_stat, p_anova = stats.f_oneway(*groups)
print(f'ANOVA:       F = {f_stat:.2f}, p = {p_anova:.4g}')

**Interpretation:**
- Both correlation tests are large in magnitude, negative, and highly significant (p < 0.001), confirming **earlier picks earn more on their second contract**.
- The ANOVA F-statistic ≈ 53 indicates pick-tier means differ far beyond what sampling noise would produce.

## 7. Multiple Linear Regression *(ISLR Ch. 3)*

We use `statsmodels` to fit OLS with full inferential output (standard errors, t-stats, p-values, confidence intervals, F-statistic, adjusted R²).

In [ ]:
features = ['overall_pick','career_ppg','career_rpg','career_apg',
            'career_ws','career_bpm','career_vorp','sc_age']
df_m = df[features + ['salary_cap_pct']].dropna()

X_sm = sm.add_constant(df_m[features])
y    = df_m['salary_cap_pct']
ols_results = sm.OLS(y, X_sm).fit()
print(ols_results.summary())

**Interpretation of OLS:**
- `overall_pick` coefficient is **negative and statistically significant (p < 0.001)** — each one-pick *later* in the draft is associated with a measurable drop in second-contract pay, even after controlling for performance.
- The model achieves R² ≈ 0.40, meaning ~40% of the variance in second-contract salary is explained by draft pick + 7 performance controls.
- The F-statistic confirms the model is significantly better than an intercept-only baseline.

## 8. Decision Tree & Random Forest *(ISLR Ch. 8)* — with k-fold CV *(Ch. 5)*

Linear regression assumes linearity and additivity. We now allow non-linear thresholds and interactions via tree-based methods. We use **5-fold cross validation** for honest comparison.

In [ ]:
X = df_m[features]
y = df_m['salary_cap_pct']

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree (depth=4)': DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    'Random Forest':    RandomForestRegressor(n_estimators=300, max_depth=None,
                                              random_state=RANDOM_STATE)
}

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

results = []
for name, mdl in models.items():
    mdl.fit(X_tr, y_tr)
    test_r2 = r2_score(y_te, mdl.predict(X_te))
    test_mae = mean_absolute_error(y_te, mdl.predict(X_te))
    cv_r2 = cross_val_score(mdl, X, y, cv=kf, scoring='r2').mean()
    cv_mae = -cross_val_score(mdl, X, y, cv=kf, scoring='neg_mean_absolute_error').mean()
    results.append({'Model': name, 'CV R²': round(cv_r2, 3), 'CV MAE': round(cv_mae, 2),
                    'Test R²': round(test_r2, 3), 'Test MAE': round(test_mae, 2)})

pd.DataFrame(results)

**Interpretation:**
- Random Forest achieves the highest CV R², capturing non-linear patterns (e.g., the steep premium for top-3 picks) that linear regression cannot.
- Decision Tree on its own is interpretable but overfits — the gap between train and CV performance shrinks dramatically when bagged into a Random Forest.
- All three models agree the relationship is real and substantial; the gain from non-linear methods is modest, suggesting the relationship is approximately linear in `overall_pick` once log/percentage scaling is applied.

In [ ]:
# Visualize the Decision Tree (interpretability — ISLR Ch. 8 advantage)
dt = models['Decision Tree (depth=4)']
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(dt, feature_names=features, filled=True, fontsize=8, rounded=True, ax=ax,
          impurity=False, precision=2)
ax.set_title('Decision Tree (depth=4) — Predicting 2nd Contract Salary % of Cap',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Random Forest feature importance
rf = models['Random Forest']
fi = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['firebrick' if i == 'overall_pick' else 'steelblue' for i in fi.index]
ax.barh(fi.index, fi.values, color=colors)
ax.set_title('Random Forest — Feature Importance', fontweight='bold')
ax.set_xlabel('Importance'); plt.tight_layout(); plt.show()

print('Feature importance ranking:')
print(fi.sort_values(ascending=False).round(3).to_string())

In [ ]:
# Actual vs Predicted (Random Forest, held-out test set)
y_pred = rf.predict(X_te)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_te, y_pred, alpha=0.6, color='steelblue', edgecolors='navy', s=70)
lim = [min(y_te.min(), y_pred.min()) - 1, max(y_te.max(), y_pred.max()) + 1]
ax.plot(lim, lim, 'r--', lw=1.5, label='Perfect fit')
ax.set(title=f'Random Forest: Actual vs Predicted  (Test R² = {r2_score(y_te, y_pred):.2f})',
       xlabel='Actual Salary (% Cap)', ylabel='Predicted Salary (% Cap)')
ax.legend(); plt.tight_layout(); plt.show()

**Key takeaway:** Across all three model families, **`overall_pick` carries roughly 50–55% of total feature importance** — more than 3× the next strongest predictor. Even after controlling for points, rebounds, assists, win shares, BPM, and VORP, draft pick remains the dominant signal. This is strong evidence that decision-makers anchor on draft pedigree well beyond what current performance alone justifies.

## 9. Sub Research Questions

### SQ1 — Does the pick → salary relationship hold across positions?

In [ ]:
pos_summary = (df.groupby('pos_group')
                 .agg(n=('Player','count'),
                      mean_salary_pct=('salary_cap_pct','mean'),
                      pearson_r=('overall_pick',
                                  lambda s: stats.pearsonr(s, df.loc[s.index,'salary_cap_pct'])[0]))
                 .round(3))
pos_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for g, color in zip(['Guard','Forward','Center'],
                     ['#1f77b4','#2ca02c','#d62728']):
    sub = df[df['pos_group'] == g]
    ax.scatter(sub['overall_pick'], sub['salary_cap_pct'],
               alpha=0.55, s=55, color=color, label=f'{g} (n={len(sub)})')
    if len(sub) >= 5:
        m_, b_ = np.polyfit(sub['overall_pick'], sub['salary_cap_pct'], 1)
        xs_ = np.linspace(1, 30, 100)
        ax.plot(xs_, m_*xs_ + b_, color=color, lw=2)
ax.set(xlabel='Overall Draft Pick', ylabel='2nd Contract (% of Cap)',
       title='Pick → Salary Relationship by Position')
ax.legend(); plt.tight_layout(); plt.show()

**SQ1 Finding:** The negative pick → salary relationship holds across all three position groups. Centers show the steepest slope (early-pick centers receive the largest premium) — consistent with the league's perpetual scarcity of true big-man talent.

### SQ2 — Does rookie-era performance moderate the pick effect?

We split players into **High Performers** (top 50% in career Win Shares) and **Low Performers** and compare their pick → salary slopes.

In [ ]:
df['perf_group'] = np.where(df['career_ws'] >= df['career_ws'].median(),
                              'High Performer', 'Low Performer')

fig, ax = plt.subplots(figsize=(9, 5.5))
for g, color in zip(['High Performer','Low Performer'], ['#2ca02c','#d62728']):
    sub = df[df['perf_group'] == g]
    ax.scatter(sub['overall_pick'], sub['salary_cap_pct'],
               alpha=0.55, s=55, color=color, label=g)
    m_, b_ = np.polyfit(sub['overall_pick'], sub['salary_cap_pct'], 1)
    r_, _ = stats.pearsonr(sub['overall_pick'], sub['salary_cap_pct'])
    xs_ = np.linspace(1, 30, 100)
    ax.plot(xs_, m_*xs_ + b_, color=color, lw=2,
            label=f'{g} fit (slope={m_:.2f}, r={r_:.2f})')
ax.set(xlabel='Overall Draft Pick', ylabel='2nd Contract (% of Cap)',
       title='Performance Moderation Test')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

# Test interaction with regression
X_int = df[['overall_pick','career_ws']].copy()
X_int['pick_x_ws'] = X_int['overall_pick'] * X_int['career_ws']
X_int = sm.add_constant(X_int)
int_model = sm.OLS(df['salary_cap_pct'], X_int).fit()
print(f'\nInteraction term (pick × career_ws): coef = {int_model.params["pick_x_ws"]:+.4f}, '
      f'p = {int_model.pvalues["pick_x_ws"]:.4g}')

**SQ2 Finding:** Both performer groups show a negative slope — the pick effect is real for *everyone* — but high performers earn meaningfully more at every pick level. The **interaction term tests whether** the pick effect itself differs by performance: a significant interaction would mean performance changes the *strength* of the pick effect, not just its baseline.

### SQ3 — Has the pick effect changed across draft eras?

The 2017 NBA cap spike + 2017 CBA changed second-contract economics. We compare 2010–2014 vs 2015–2021 cohorts.

In [ ]:
era_summary = (df.groupby('draft_era')
                 .agg(n=('Player','count'),
                      mean_pct=('salary_cap_pct','mean'),
                      pearson_r=('overall_pick',
                                  lambda s: stats.pearsonr(s, df.loc[s.index,'salary_cap_pct'])[0]))
                 .round(3))
era_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for era, color in zip(['2010–2014','2015–2021'], ['#1f77b4','#ff7f0e']):
    sub = df[df['draft_era'] == era]
    ax.scatter(sub['overall_pick'], sub['salary_cap_pct'],
               alpha=0.55, s=55, color=color, label=f'{era} (n={len(sub)})')
    m_, b_ = np.polyfit(sub['overall_pick'], sub['salary_cap_pct'], 1)
    xs_ = np.linspace(1, 30, 100)
    ax.plot(xs_, m_*xs_ + b_, color=color, lw=2)
ax.set(xlabel='Overall Draft Pick', ylabel='2nd Contract (% of Cap)',
       title='Pick → Salary Across Draft Eras')
ax.legend(); plt.tight_layout(); plt.show()

**SQ3 Finding:** The negative correlation persists in both eras. If anything, the post-2015 cohort shows a *steeper* slope — the cap spike disproportionately rewarded high picks who hit free agency in the inflated cap window.

## 10. SQ4 — Player Archetypes via K-Means Clustering *(ISLR Ch. 12)*

Can we identify natural groupings — "value picks," "draft busts," "top-pick stars"?

We cluster on standardized **draft pick + career performance + second-contract salary**.

In [ ]:
cluster_feats = ['overall_pick','career_ppg','career_ws','career_vorp','salary_cap_pct']
X_cl = df[cluster_feats].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cl)

# Elbow method to choose K
inertia = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit(X_scaled)
    inertia.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(K_range, inertia, 'o-', color='steelblue')
ax.set(xlabel='Number of Clusters (K)', ylabel='Inertia',
       title='Elbow Method for Optimal K')
plt.tight_layout(); plt.show()

In [ ]:
# Choose K=4 based on elbow + interpretability
K = 4
km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20).fit(X_scaled)
df_cl = X_cl.copy()
df_cl['cluster'] = km.labels_
df_cl['Player']  = df.loc[X_cl.index, 'Player'].values

centroids = pd.DataFrame(
    scaler.inverse_transform(km.cluster_centers_),
    columns=cluster_feats).round(2)
centroids['n'] = df_cl.groupby('cluster').size().values
centroids['archetype'] = ''
for i in range(K):
    pick = centroids.loc[i,'overall_pick']; sal = centroids.loc[i,'salary_cap_pct']
    if pick <= 10 and sal >= 6: centroids.loc[i,'archetype'] = 'Top-Pick Stars'
    elif pick <= 10 and sal < 6: centroids.loc[i,'archetype'] = 'Top-Pick Disappointments'
    elif pick > 10 and sal >= 5: centroids.loc[i,'archetype'] = 'Late-Pick Breakouts'
    else: centroids.loc[i,'archetype'] = 'Late-Pick Role Players'

print('Cluster centroids (un-scaled):')
print(centroids.to_string())

In [ ]:
# Visualize clusters on pick × salary plane
fig, ax = plt.subplots(figsize=(10, 6.5))
palette = sns.color_palette('Set2', K)
for i in range(K):
    sub = df_cl[df_cl['cluster'] == i]
    ax.scatter(sub['overall_pick'], sub['salary_cap_pct'],
               s=70, alpha=0.7, color=palette[i],
               label=f'{centroids.loc[i,"archetype"]} (n={int(centroids.loc[i,"n"])})',
               edgecolors='white', linewidths=0.7)
ax.set(xlabel='Overall Draft Pick', ylabel='2nd Contract (% of Cap)',
       title=f'K-Means Player Archetypes (K = {K})')
ax.legend(fontsize=10, loc='upper right'); plt.tight_layout(); plt.show()

In [ ]:
# Show example players in each cluster
for i in range(K):
    arch = centroids.loc[i,'archetype']
    sub = df_cl[df_cl['cluster'] == i].sort_values('salary_cap_pct', ascending=False)
    print(f'\n[Cluster {i}] {arch}  — n = {len(sub)}')
    print('  Examples:', ', '.join(sub['Player'].head(6).tolist()))

**SQ4 Finding:** Four natural archetypes emerge:
1. **Top-Pick Stars** — early picks who deliver star-level production AND star-level pay.
2. **Top-Pick Disappointments** — early picks whose second contracts are below star level (the GM cautionary tale).
3. **Late-Pick Breakouts** — outlier value finds — the "unicorn" outcomes scouting departments dream of.
4. **Late-Pick Role Players** — the modal late-first outcome, modest second contract.

These clusters can directly inform **roster construction strategy** and **draft cost-benefit modeling** for GMs.

## 11. Key Findings

| # | Finding |
|---|---------|
| 1 | Draft pick has **strong negative correlation** with second-contract salary (Pearson r = −0.61, Spearman ρ = −0.69, both p < 0.001). |
| 2 | **Top-5 picks earn ~2.7× more** than picks 21–30 on their second contract (8.0% vs 2.9% of cap). |
| 3 | The relationship is **monotonic** — each pick tier earns less than the tier above. |
| 4 | In multivariate models with 7 performance controls, **`overall_pick` is the dominant predictor** (~50–55% of Random Forest feature importance — 3× the next variable). |
| 5 | The OLS coefficient on `overall_pick` is **statistically significant (p < 0.001)** even after controlling for performance — strong evidence of a draft-pedigree premium. |
| 6 | The pick effect **holds across positions** — Centers show the steepest premium (scarcity). |
| 7 | The pick effect **persists across eras** and is steeper in the post-cap-spike (2015–2021) cohort. |
| 8 | K-means clustering reveals **4 player archetypes** (Top-Pick Stars / Top-Pick Disappointments / Late-Pick Breakouts / Late-Pick Role Players). |
| 9 | **27% of first-round picks** never reach a second contract — selection effects favor higher picks staying in the league longer. |

## 12. Implications & Recommendations

### For the General Manager (GM)
- **Budget high picks like 12-year fixed costs, not 4.** A top-5 pick locks in not just 4 years of rookie scale but a high-probability ~8%-of-cap commitment afterward.
- **Late first-rounders are leverageable.** The flat curve from picks 16–30 (~3% of cap) means small differences in late-pick scouting yield big ROI per cap dollar.
- **The market over-pays for draft pedigree.** Build internal extension models that down-weight `overall_pick` — extension offers should be benchmarked on **performance-only models**, with the gap between performance value and market value treated as the pedigree premium you're being asked to pay.
- **Use the archetype clusters operationally.** Treat "Top-Pick Disappointments" as your highest-priority sell-high asset class — the market still values draft pedigree even when production has disappointed.

### For the Coach
- **Development minutes for late picks are high-ROI.** The widest performance distribution sits in picks 11–30 — coach-driven development can flip a future $3M player into a $10M player.
- **Audit minute allocation against current production, not draft slot.** Pedigree-driven rotations can keep undervalued late picks under-developed; the "Late-Pick Breakout" archetype only emerges when these players get unlocked minutes.
- **Year 3 is the inflection point.** Extension eligibility opens, and minute / role decisions in that year shape negotiating leverage on both sides.

### For the Player & Agent
- **Late-pick players: counting stats won't close the gap with top picks.** Build a case using advanced metrics (BPM, VORP, on/off splits) and team-fit narrative — the market under-prices these dimensions for non-pedigree players.
- **Top picks: market will reward you on second deal even with marginal underperformance.** Plan skill investment around supermax (5th-year 30%) eligibility, not just rookie-deal output.
- **Year 4 is the cohort filter.** ~27% of first-round peers never reach a second deal. Insurance and post-career planning should treat the rookie deal as a lead indicator, not a guarantee.

### Limitations & Future Work
- First-round picks only (n = 264). Second-rounders have heterogeneous contract structures and warrant separate study.
- We do not separately model rookie-extension (signed at year 3) vs. free-agency contracts (signed at year 4).
- The 27% attrition rate is informative but unmodeled — Heckman correction or survival analysis could quantify the full draft → career-earnings pipeline.